# 1. 환경 설정 및 데이터 로드

In [2]:
from pathlib import Path

print("현재 작업 디렉터리:", Path.cwd())
print("현재 폴더의 파일:", [p.name for p in Path.cwd().iterdir()])

현재 작업 디렉터리: /Users/admin/Documents/ah07-cancer-subtype/notebooks/H_data
현재 폴더의 파일: ['07_subtype_feature_full_ablation.ipynb', 'cache', '05_mutation_signature.ipynb', '08_twin_fold_comparison.ipynb', '02_features.ipynb', '01_eda.ipynb', '04_position_bin_tuning.ipynb', '06_subtype_feature_ablation.ipynb', '00_train_test_distribution.ipynb', '03_hotspot_bins.ipynb']


In [3]:

import sys
from pathlib import Path

import numpy as np
import pandas as pd

# notebooks/H_data → 프로젝트 루트
ROOT = Path.cwd().resolve().parents[1]

assert (ROOT / "src").is_dir(), "프로젝트 루트를 찾을 수 없습니다."

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import DATA_DIR
from src.data_loader import load_data, split_xy, encode_target
from src.validation import get_folds, twin_groups

train, test = load_data(DATA_DIR)
X_raw, y_raw = split_xy(train)
y, le = encode_target(y_raw)

print("프로젝트 루트:", ROOT)
print("전체 학습 데이터:", X_raw.shape)
print("타깃 클래스 수:", len(le.classes_))
print("라벨 개수:", len(y))
print("데이터 인덱스:", type(X_raw.index).__name__)



프로젝트 루트: /Users/admin/Documents/ah07-cancer-subtype
전체 학습 데이터: (6201, 4384)
타깃 클래스 수: 26
라벨 개수: 6201
데이터 인덱스: RangeIndex


In [5]:
import json

path = ROOT / "notebooks/A_model/10_lgbm_g3_3model_equal_ensemble.ipynb"

with path.open(encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    for line in source.splitlines():
        if "holdout_module" in line or "importlib" in line:
            print(f"셀 {i}: {line.strip()}")

셀 1: import importlib.util
셀 1: spec = importlib.util.spec_from_file_location('fixed_holdout_manifest', manifest_path)
셀 1: holdout_module = importlib.util.module_from_spec(spec)
셀 1: spec.loader.exec_module(holdout_module)
셀 1: assert holdout_module.ROOT.resolve() == ROOT.resolve(), '스크립트의 ROOT 경로를 확인하세요.'
셀 3: dev_idx, hold_idx = holdout_module.verify()


In [6]:
# 기존 팀 노트북의 셀 1에서 Holdout 관련 설정 확인
print("".join(nb["cells"][1]["source"]))

from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p/'src'/'config.py').is_file() and (p/'scripts'/'fixed_holdout_manifest.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('src/config.py가 있는 프로젝트 폴더에서 실행하세요.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 노트북의 현재 작업 폴더와 무관하게 프로젝트 스크립트 로딩
import importlib.util
manifest_path = ROOT / 'scripts' / 'fixed_holdout_manifest.py'
spec = importlib.util.spec_from_file_location('fixed_holdout_manifest', manifest_path)
holdout_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(holdout_module)
assert holdout_module.ROOT.resolve(

In [7]:

import importlib.util

# 팀 공통 Holdout 스크립트 경로
manifest_path = ROOT / "scripts" / "fixed_holdout_manifest.py"

assert manifest_path.is_file(), "Holdout 스크립트를 찾을 수 없습니다."

# 기존 팀 노트북과 동일한 방식으로 로드
spec = importlib.util.spec_from_file_location(
    "fixed_holdout_manifest",
    manifest_path
)
holdout_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(holdout_module)

assert holdout_module.ROOT.resolve() == ROOT.resolve()

# 팀에서 고정한 Dev / Holdout 인덱스
dev_idx, hold_idx = holdout_module.verify()

assert len(dev_idx) + len(hold_idx) == len(y)
assert len(np.intersect1d(dev_idx, hold_idx)) == 0

# Dev 데이터 구성
X_dev_raw = X_raw.iloc[dev_idx].reset_index(drop=True)
y_dev = np.asarray(y)[dev_idx]

print("Dev 데이터:", X_dev_raw.shape)
print("Dev 라벨:", y_dev.shape)
print("Holdout 데이터:", len(hold_idx))
print("Dev 클래스 수:", len(np.unique(y_dev)))


VERIFIED: dev=4960, holdout=1241; same IDs and row signatures
Dev 데이터: (4960, 4384)
Dev 라벨: (4960,)
Holdout 데이터: 1241
Dev 클래스 수: 26


# 2. 피처 구성방법 E(아미노산 치환 쌍 + 유전자별 끊김형) 기반 쌍둥이 Fold 비교 실험

In [8]:
# 쌍둥이 그룹 및 환자 수 확인

# 팀 공통 함수로 쌍둥이 그룹 생성
groups = twin_groups(
    X_dev_raw,
    y_dev,
    classes=le.classes_
)

# 같은 그룹 ID를 공유하는 환자 수 계산
group_ids, group_counts = np.unique(
    groups,
    return_counts=True
)

# 2명 이상 포함된 그룹만 쌍둥이 그룹으로 정의
twin_group_ids = group_ids[group_counts >= 2]

# 환자별 쌍둥이 여부
twin_mask = np.isin(groups, twin_group_ids)
non_twin_mask = ~twin_mask

print("전체 Dev 환자:", len(y_dev))
print("쌍둥이 그룹 수:", len(twin_group_ids))
print("쌍둥이 환자 수:", twin_mask.sum())
print("비쌍둥이 환자 수:", non_twin_mask.sum())
print("쌍둥이 환자 비율:", f"{twin_mask.mean():.2%}")

# 그룹별 환자 수 분포
print("\n쌍둥이 그룹 크기 분포:")
print(
    pd.Series(group_counts[group_counts >= 2])
    .value_counts()
    .sort_index()
)


전체 Dev 환자: 4960
쌍둥이 그룹 수: 345
쌍둥이 환자 수: 690
비쌍둥이 환자 수: 4270
쌍둥이 환자 비율: 13.91%

쌍둥이 그룹 크기 분포:
2    345
Name: count, dtype: int64


In [9]:
# Fold별 쌍둥이 분리 현황 비교


from sklearn.model_selection import StratifiedKFold

# 1. 팀 공통 Fold: 쌍둥이 그룹 유지
team_folds = get_folds(
    y_dev,
    X=X_dev_raw,
    classes=le.classes_
)

# 2. 일반 StratifiedKFold: 쌍둥이 그룹 고려하지 않음
skf_folds = {}

for seed in [0, 1]:
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed
    )
    skf_folds[seed] = list(skf.split(X_dev_raw, y_dev))

# 3. 각 Fold에서 학습/검증으로 분리된 쌍둥이 환자 수
def count_split_twins(folds, groups, twin_mask):
    split_count = 0

    for train_idx, val_idx in folds:
        train_groups = set(groups[train_idx])

        split_count += sum(
            twin_mask[i] and groups[i] in train_groups
            for i in val_idx
        )

    return split_count

fold_summary = pd.DataFrame([
    {
        "Fold 방식": "Team GroupFold",
        "분리된 쌍둥이 환자": count_split_twins(
            team_folds, groups, twin_mask
        )
    },
    *[
        {
            "Fold 방식": f"StratifiedKFold seed={seed}",
            "분리된 쌍둥이 환자": count_split_twins(
                skf_folds[seed], groups, twin_mask
            )
        }
        for seed in [0, 1]
    ]
])

fold_summary["쌍둥이 중 분리 비율"] = (
    fold_summary["분리된 쌍둥이 환자"] / twin_mask.sum()
)

display(fold_summary)


,Fold 방식,분리된 쌍둥이 환자,쌍둥이 중 분리 비율
0,Team GroupFold,0,0.000000
1,StratifiedKFold seed=0,526,0.762319
2,StratifiedKFold seed=1,540,0.782609


In [11]:
# 피처 구성방법 E 기반 쌍둥이 Fold 비교 실험


from src.preprocessing import binarize_mutations
from src.features import (
    mutation_count,
    mutation_type_counts,
    amino_acid_pair_counts,
    gene_truncation_flags,
)

from scipy.sparse import csr_matrix

def make_E(X):
    # 기존 팀 Feature E와 동일한 구성 및 순서
    parts = [
        binarize_mutations(X),
        np.log1p(mutation_count(X)),
        np.log1p(mutation_type_counts(X)),
        np.log1p(amino_acid_pair_counts(X)),
        gene_truncation_flags(X),
    ]

    out = pd.concat(parts, axis=1).astype(np.float32)

    assert not out.columns.has_duplicates
    assert out.index.equals(X.index)
    assert np.isfinite(out.to_numpy()).all()

    return out


# Dev 데이터의 Feature E 생성
X_dev_E = make_E(X_dev_raw)

assert X_dev_E.shape == (4960, 9174)

# LR 입력용 희소행렬
X_dev_E_sparse = csr_matrix(
    X_dev_E.to_numpy(dtype=np.float32)
)

print("Feature E:", X_dev_E.shape)
print("희소행렬:", X_dev_E_sparse.shape)
print("희소행렬 타입:", type(X_dev_E_sparse).__name__)
print("결측값 수:", X_dev_E.isna().sum().sum())


Feature E: (4960, 9174)
희소행렬: (4960, 9174)
희소행렬 타입: csr_matrix
결측값 수: 0


In [12]:
# 동일한 LR 모델로 OOF 예측 비교


from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
import time

# 기존 팀 LR 설정
base_lr = LogisticRegression(
    C=0.1,
    class_weight="balanced",
    solver="lbfgs",
    max_iter=3000
)

fold_configs = {
    "Team GroupFold": team_folds,
    "StratifiedKFold seed=0": skf_folds[0],
    "StratifiedKFold seed=1": skf_folds[1],
}

oof_predictions = {}
cv_results = []

for name, folds in fold_configs.items():
    start_time = time.time()
    oof_pred = np.full(len(y_dev), -1, dtype=int)

    for fold_num, (train_idx, val_idx) in enumerate(folds, start=1):
        model = clone(base_lr)

        model.fit(
            X_dev_E_sparse[train_idx],
            y_dev[train_idx]
        )

        oof_pred[val_idx] = model.predict(
            X_dev_E_sparse[val_idx]
        )

        print(f"{name} | Fold {fold_num}/5 완료")

    assert np.all(oof_pred >= 0), "OOF 예측 누락 발생"

    oof_predictions[name] = oof_pred

    cv_results.append({
        "Fold 방식": name,
        "OOF Macro F1": f1_score(
            y_dev, oof_pred, average="macro"
        ),
        "쌍둥이 Accuracy": accuracy_score(
            y_dev[twin_mask], oof_pred[twin_mask]
        ),
        "비쌍둥이 Accuracy": accuracy_score(
            y_dev[non_twin_mask], oof_pred[non_twin_mask]
        ),
        "실행 시간(초)": round(time.time() - start_time, 1)
    })

results_df = pd.DataFrame(cv_results)

display(results_df.round(4))


Team GroupFold | Fold 1/5 완료
Team GroupFold | Fold 2/5 완료
Team GroupFold | Fold 3/5 완료
Team GroupFold | Fold 4/5 완료
Team GroupFold | Fold 5/5 완료
StratifiedKFold seed=0 | Fold 1/5 완료
StratifiedKFold seed=0 | Fold 2/5 완료
StratifiedKFold seed=0 | Fold 3/5 완료
StratifiedKFold seed=0 | Fold 4/5 완료
StratifiedKFold seed=0 | Fold 5/5 완료
StratifiedKFold seed=1 | Fold 1/5 완료
StratifiedKFold seed=1 | Fold 2/5 완료
StratifiedKFold seed=1 | Fold 3/5 완료
StratifiedKFold seed=1 | Fold 4/5 완료
StratifiedKFold seed=1 | Fold 5/5 완료


,Fold 방식,OOF Macro F1,쌍둥이 Accuracy,비쌍둥이 Accuracy,실행 시간(초)
0,Team GroupFold,0.4849,0.3681,0.5042,13.8
1,StratifiedKFold seed=0,0.4652,0.1986,0.5000,13.8
2,StratifiedKFold seed=1,0.4646,0.1913,0.4974,14.1


- 일반 SKFold에서 성능이 떨어진 현상은 쌍둥이 환자에서 특히 두드러짐.
- 다만 이 결과만으로 CV–LB 차이 0.11 중 정확히 0.02가 쌍둥이 때문이라고 단정하면 안됨. Fold를 변경하면서 학습 데이터 구성도 함께 달라졌기 때문임.

In [13]:
# 충돌 암종 4개의 F1 계산


from sklearn.metrics import f1_score

target_classes = ["KIPAN", "KIRC", "GBMLGG", "LGG"]

# 기존 LabelEncoder의 클래스 순서 사용
target_indices = [
    int(le.transform([name])[0])
    for name in target_classes
]

class_f1_results = []

for fold_name, oof_pred in oof_predictions.items():
    class_f1 = f1_score(
        y_dev,
        oof_pred,
        labels=target_indices,
        average=None,
        zero_division=0
    )

    row = {"Fold 방식": fold_name}

    for class_name, score in zip(target_classes, class_f1):
        row[f"{class_name} F1"] = score

    row["충돌 4종 평균 F1"] = np.mean(class_f1)

    class_f1_results.append(row)

class_f1_df = pd.DataFrame(class_f1_results)

display(class_f1_df.round(4))


,Fold 방식,KIPAN F1,KIRC F1,GBMLGG F1,LGG F1,충돌 4종 평균 F1
0,Team GroupFold,0.3898,0.4024,0.4340,0.5660,0.4481
1,StratifiedKFold seed=0,0.2636,0.2180,0.3575,0.4599,0.3247
2,StratifiedKFold seed=1,0.2414,0.1954,0.3615,0.4668,0.3163


In [ ]:
# 발표용 통합 결과표 생성


# 기존 실험 결과 통합
final_results = (
    results_df
    .merge(
        fold_summary,
        on="Fold 방식",
        how="left"
    )
    .merge(
        class_f1_df,
        on="Fold 방식",
        how="left"
    )
)

# 발표용 컬럼 순서
columns = [
    "Fold 방식",
    "분리된 쌍둥이 환자",
    "쌍둥이 중 분리 비율",
    "OOF Macro F1",
    "쌍둥이 Accuracy",
    "비쌍둥이 Accuracy",
    "KIPAN F1",
    "KIRC F1",
    "GBMLGG F1",
    "LGG F1",
    "충돌 4종 평균 F1",
]

final_results = final_results[columns]

display(final_results.round(4))


,Fold 방식,분리된 쌍둥이 환자,쌍둥이 중 분리 비율,OOF Macro F1,쌍둥이 Accuracy,비쌍둥이 Accuracy,KIPAN F1,KIRC F1,GBMLGG F1,LGG F1,충돌 4종 평균 F1
0,Team GroupFold,0,0.0000,0.4849,0.3681,0.5042,0.3898,0.4024,0.4340,0.5660,0.4481
1,StratifiedKFold seed=0,526,0.7623,0.4652,0.1986,0.5000,0.2636,0.2180,0.3575,0.4599,0.3247
2,StratifiedKFold seed=1,540,0.7826,0.4646,0.1913,0.4974,0.2414,0.1954,0.3615,0.4668,0.3163


# 3. 실험 결과 및 해석

### 1. 실험 목적

동일한 유전자 변이 정보를 가지지만 암종 라벨이 서로 다른
쌍둥이 충돌 샘플이 CV 성능에 미치는 영향을 확인한다.

특히 팀 공통 GroupFold와 일반 StratifiedKFold를 비교하여
CV 분할 방식에 따른 성능 민감도를 분석한다.

### 2. 실험 조건

- 데이터: Dev 4,960건 (고정 Holdout 1,241건 제외)
- 피처: 피처 구성방법 E (9,174개)
  - 유전자별 변이 여부
  - 변이 개수 및 변이 유형
  - 아미노산 치환 쌍
  - 유전자별 끊김형 변이 여부
- 모델: LogisticRegression
  - C=0.1
  - class_weight="balanced"
  - solver="lbfgs"
  - max_iter=3000
- 검증: 5-Fold OOF
  - Team GroupFold
  - StratifiedKFold(seed=0)
  - StratifiedKFold(seed=1)

### 3. 쌍둥이 충돌 샘플 현황

- 쌍둥이 그룹: 345개
- 쌍둥이 환자: 690명 (13.91%)
- 비쌍둥이 환자: 4,270명
- 쌍둥이 그룹 크기: 모두 2명

일반 StratifiedKFold에서는 쌍둥이 환자의
76.23~78.26%가 학습/검증 Fold로 분리되었다.

### 4. 주요 결과

| 지표 | Team GroupFold | SKFold seed=0 | SKFold seed=1 |
|---|---:|---:|---:|
| 분리된 쌍둥이 환자 | 0 | 526 | 540 |
| OOF Macro F1 | 0.4849 | 0.4652 | 0.4646 |
| 쌍둥이 Accuracy | 0.3681 | 0.1986 | 0.1913 |
| 비쌍둥이 Accuracy | 0.5042 | 0.5000 | 0.4974 |
| KIPAN F1 | 0.3898 | 0.2636 | 0.2414 |
| KIRC F1 | 0.4024 | 0.2180 | 0.1954 |
| GBMLGG F1 | 0.4340 | 0.3575 | 0.3615 |
| LGG F1 | 0.5660 | 0.4599 | 0.4668 |
| 충돌 4종 평균 F1 | 0.4481 | 0.3247 | 0.3163 |

### 5. 결과 해석

**① 전체 Macro F1**

일반 StratifiedKFold의 평균 OOF Macro F1은
0.4649로, Team GroupFold(0.4849) 대비
약 0.0200 낮았다.

**② 쌍둥이 환자 정확도**

쌍둥이 Accuracy는 0.3681에서 평균 0.1949로
약 17.32%p 하락했다.

반면 비쌍둥이 Accuracy는 0.5042에서 평균
0.4987로 약 0.55%p 하락했다.

**③ 충돌 암종 4종의 F1**

KIPAN, KIRC, GBMLGG, LGG의 평균 F1은
0.4481에서 평균 0.3205로 약 0.1276 하락했다.

특히 KIRC와 KIPAN에서 큰 성능 저하가 관찰되었다.

### 6. 결론

쌍둥이 충돌 샘플을 서로 다른 학습/검증 Fold로
분리하면 전체 Macro F1이 약 0.020 하락했다.

이러한 성능 저하는 비쌍둥이 환자보다
쌍둥이 환자와 충돌 암종에서 뚜렷하게 나타났다.

따라서 쌍둥이 충돌 구조는 CV 분할 방식에 따른
성능 차이에 영향을 주는 요인으로 판단된다.

### 7. 해석의 한계

- Fold 변경에 따른 학습 데이터 구성 차이가 함께 존재한다.
- StratifiedKFold의 결과는 seed 0, 1 두 번만 비교했다.
- 이번 실험은 CV 분할 방식의 민감도를 분석한 것이며,
  CV-LB 차이 0.11 중 쌍둥이의 정확한 기여도를
  인과적으로 분리한 실험은 아니다.
- 따라서 Macro F1 차이 약 0.020을
  CV-LB 격차에서 쌍둥이가 차지하는 정확한 몫으로
  해석해서는 안 된다.
